In [1]:
import tkinter as tk
import random
import time

GOAL = (1, 2, 3, 4, 5, 6, 7, 8, 0)
TILE = 90
ANIM = 300

GOAL_POS = {val: divmod(i, 3) for i, val in enumerate(GOAL)}
def h_misplaced(board):
    return sum(1 for i, val in enumerate(board) if val != 0 and val != GOAL[i])


def manhattan(board):
    total = 0
    for i, val in enumerate(board):
        if val == 0:
            continue
        r, c = divmod(i, 3)
        gr, gc = GOAL_POS[val]
        total += abs(r - gr) + abs(c - gc)
    return total

def get_neighbors(board):
    idx = board.index(0)
    r, c = divmod(idx, 3)
    result = []
    for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
        nr, nc = r + dr, c + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            ni = nr * 3 + nc
            temp = list(board)
            temp[idx], temp[ni] = temp[ni], temp[idx]
            result.append(tuple(temp))
    return result


def get_direction(old_state, new_state):
    diff = new_state.index(0) - old_state.index(0)
    return {-3: "UP", 3: "DOWN", -1: "LEFT", 1: "RIGHT"}.get(diff, "?")


def is_solvable(board):
    arr = [x for x in board if x != 0]
    inv = sum(1 for i in range(len(arr)) for j in range(i + 1, len(arr)) if arr[i] > arr[j])
    return inv % 2 == 0


def random_board():
    b = list(GOAL)
    while True:
        random.shuffle(b)
        t = tuple(b)
        if t != GOAL and is_solvable(t):
            return t

def ida_star(start):
    threshold = h_misplaced(start)
    path = [start]
    total_nodes = [0]

    def search(g):
        nonlocal threshold
        current = path[-1]
        f = g + h_misplaced(current)
        total_nodes[0] += 1
        if f > threshold:
            return f        
        if current == GOAL:
            return "FOUND"

        minimum = float('inf')
        for child in get_neighbors(current):
            if child not in path:
                path.append(child)
                result = search(g + 1)
                if result == "FOUND":
                    return "FOUND"
                if result < minimum:
                    minimum = result
                path.pop()

        return minimum

    while True:
        result = search(0)
        if result == "FOUND":
            return list(path), total_nodes[0]
        if result == float('inf'):
            return None, total_nodes[0]
        threshold = result

class App:

    def __init__(self, root):
        self.root = root
        self.root.title("8 Puzzle - IDA*")
        self.root.configure(bg="white")
        self.root.resizable(False, False)

        self.board = random_board()
        self.solution = []
        self.moves = []
        self.step = 0
        self.playing = False
        main = tk.Frame(root, bg="white")
        main.pack(padx=10, pady=10)
        left = tk.Frame(main, bg="white")
        left.pack(side="left", padx=10)

        self.canvas = tk.Canvas(
            left, width=3 * TILE, height=3 * TILE,
            bg="white", highlightthickness=0
        )
        self.canvas.pack()
        self.step_label = tk.Label(left, text="Steps:", font=("Arial", 12), bg="white")
        self.step_label.pack(pady=6)
        self.lb_h = tk.Label(left, text="h(n): 0", bg="white", fg="blue", font=("Arial", 11))
        self.lb_h.pack()
        info = tk.Frame(left, bg="white")
        info.pack(pady=4)
        self.lb_nodes = tk.Label(info, text="Nodes: 0", bg="white")
        self.lb_nodes.grid(row=0, column=0, padx=6)
        self.lb_steps = tk.Label(info, text="Steps: 0", bg="white")
        self.lb_steps.grid(row=0, column=1, padx=6)
        self.lb_time = tk.Label(info, text="Time: 0 ms", bg="white")
        self.lb_time.grid(row=0, column=2, padx=6)
        btns = tk.Frame(left, bg="white")
        btns.pack(pady=6)

        tk.Button(btns, text="Shuffle", width=10, command=self.shuffle).grid(row=0, column=0, padx=4)
        tk.Button(btns, text="Reset",   width=10, command=self.reset  ).grid(row=0, column=1, padx=4)
        tk.Button(btns, text="Solve",   width=10, command=self.solve  ).grid(row=0, column=2, padx=4)
        tk.Button(btns, text="Step", width=34, command=self.next_step).grid(
            row=1, column=0, columnspan=3, pady=4
        )

        right = tk.Frame(main, bg="white")
        right.pack(side="right", padx=10)
        tk.Label(right, text="Solution", font=("Arial", 14, "bold"), bg="white").pack()
        self.solution_box = tk.Text(right, width=22, height=20, font=("Consolas", 10))
        self.solution_box.pack()
        self.draw()

    def current_board(self):
        if self.solution:
            return self.solution[self.step]
        return self.board

    def draw(self):
        self.canvas.delete("all")
        board = self.current_board()
        for i, val in enumerate(board):
            r, c = divmod(i, 3)
            x1, y1 = c * TILE, r * TILE
            x2, y2 = x1 + TILE, y1 + TILE
            if val == 0:
                self.canvas.create_rectangle(
                    x1, y1, x2, y2, fill="#f2f2f2", outline="black", width=2
                )
            else:
                wrong = (val != GOAL[i])
                self.canvas.create_rectangle(
                    x1, y1, x2, y2, fill="white", outline="black", width=2
                )
                self.canvas.create_text(
                    (x1 + x2) // 2, (y1 + y2) // 2,
                    text=str(val),
                    font=("Arial", 28, "bold"),
                    fill="red" if wrong else "black"
                )
        self.lb_h.config(text=f"h(n): {h_misplaced(board)}")

    def shuffle(self):
        self.playing = False
        self.board = random_board()
        self.solution = []
        self.moves = []
        self.step = 0
        self.solution_box.delete("1.0", tk.END)
        self.lb_nodes.config(text="Nodes: 0")
        self.lb_steps.config(text="Steps: 0")
        self.lb_time.config(text="Time: 0 ms")
        self.step_label.config(text="Steps:")
        self.draw()

    def reset(self):
        self.playing = False
        self.board = GOAL
        self.solution = []
        self.moves = []
        self.step = 0
        self.solution_box.delete("1.0", tk.END)
        self.draw()

    def solve(self):
        if self.playing:
            return
        t0 = time.perf_counter()
        path, nodes = ida_star(self.board)
        ms = (time.perf_counter() - t0) * 1000

        if path is None:
            self.step_label.config(text="No solution!")
            return

        self.solution = path
        self.moves = [get_direction(path[i], path[i + 1]) for i in range(len(path) - 1)]
        self.step = 0

        self.lb_nodes.config(text=f"Nodes: {nodes}")
        self.lb_steps.config(text=f"Steps: {len(path) - 1}")
        self.lb_time.config(text=f"Time: {ms:.2f} ms")
        self.solution_box.delete("1.0", tk.END)
        self.playing = True
        self.animate()

    def animate(self):
        if not self.playing:
            return

        self.draw()

        if self.step == 0:
            self.step_label.config(text="START")
        else:
            g = self.step
            h = h_misplaced(self.solution[self.step])
            self.step_label.config(
                text=f"Step {self.step}: {self.moves[self.step - 1]}   "
                     f"g={g}  h={h}  f={g + h}"
            )

        self.solution_box.tag_remove("highlight", "1.0", tk.END)
        if self.step < len(self.moves):
            h_val = h_misplaced(self.solution[self.step])
            g_val = self.step
            self.solution_box.insert(
                tk.END,
                f"{self.step}. {self.moves[self.step - 1] if self.step > 0 else '---'}"
                f"   h={h_val}  g={g_val}  f={g_val + h_val}\n"
            )

        if self.step > 0:
            line = f"{self.step}.0"
            self.solution_box.tag_add("highlight", line, f"{self.step}.end")
            self.solution_box.tag_config("highlight", background="#cce5ff")
            self.solution_box.see(line)

        if self.step < len(self.solution) - 1:
            self.step += 1
            self.root.after(ANIM, self.animate)
        else:
            self.playing = False

    def next_step(self):
        self.playing = False
        if not self.solution:
            return
        if self.step < len(self.solution) - 1:
            self.step += 1
            self.draw()
            g = self.step
            h = h_misplaced(self.solution[self.step])
            self.step_label.config(
                text=f"Step {self.step}: {self.moves[self.step - 1]}   "
                     f"g={g}  h={h}  f={g + h}"
            )


if __name__ == "__main__":
    root = tk.Tk()
    app = App(root)
    root.mainloop()